# Orion extractor server on a Kaggle GPU

Runs `python -m extractor serve` (GLiNER2 + BookNLP on GPU 0, NuExtract 2.0 8B Q8 through Ollama on GPU 1) and
publishes it through an ngrok tunnel, so the backend on your laptop extracts here instead of on its 6 GB GPU.
On the gold stories the 8B Q8 model raised facts F1 from 0.58 (local 4B) to 0.72; everything else is identical
(`kaggle_model_comparison.ipynb`).

**Once:**
1. Free ngrok account → claim your free **static domain** (dashboard → Domains) and copy your **authtoken**.
2. Kaggle → Add-ons → **Secrets**, attach to this notebook:
   - `NGROK_AUTHTOKEN`: the ngrok authtoken
   - `NGROK_DOMAIN`: the static domain, e.g. `orion-demo.ngrok-free.app`
   - `ORION_EXTRACTOR_TOKEN`: any long random string (`python -c "import secrets; print(secrets.token_urlsafe(32))"`)
3. Settings → Accelerator **GPU T4 x2**, Settings → **Internet on**.

**Each session:** *Save Version → Save & Run All (Commit)*. A committed run keeps going with the browser closed,
until the last cell's 11.5 h loop ends (Kaggle's limit is 12 h; it uses your 30 GPU h/week). Setup takes ~15 min;
the server is up when the log shows `public: {"model": ...}`.

**Backend** (`backend/.env`), then restart it:
```
EXTRACTOR_URL="https://<NGROK_DOMAIN>"
EXTRACTOR_TOKEN="<ORION_EXTRACTOR_TOKEN>"
```
Remove `EXTRACTOR_URL` to extract locally again.


In [ ]:
REPO = "https://github.com/SkySwordSaint26/Orion---World-State-Engine.git"
BRANCH = "feat/remote-extractor"            # use "main" once this branch is merged
MODEL = "nuextract2-8b-q8"                  # Ollama name, registered from (Hugging Face repo, GGUF file):
GGUF = ("numind/NuExtract-2.0-8B-GGUF", "NuExtract-2.0-8B-Q8_0.gguf")
PORT = 8000
HOURS = 11.5                                # Kaggle stops a session at 12 h
WORK = "/kaggle/working"

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
NGROK_AUTHTOKEN, NGROK_DOMAIN, TOKEN = (secrets.get_secret(k) for k in
                                        ("NGROK_AUTHTOKEN", "NGROK_DOMAIN", "ORION_EXTRACTOR_TOKEN"))

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
os.chdir(WORK)
!rm -rf orion && git clone -q --depth 1 -b {BRANCH} {REPO} orion && git -C orion log --oneline -1
EXT = f"{WORK}/orion/extractor"

## 2. Python 3.12 environment

A separate venv with the extractor's pinned versions, so Kaggle's preinstalled packages can't clash with them.

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 {WORK}/venv
PY = f"{WORK}/venv/bin/python"
!uv pip install -q -p {PY} -r {EXT}/requirements.txt
!uv pip install -q -p {PY} --no-deps booknlp==1.0.8
!uv pip install -q -p {PY} https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
# The pinned torch must see the GPU; if Kaggle's driver is too old for its CUDA build, fall back to a CUDA 12.6 build.
check = lambda: subprocess.run([PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                               capture_output=True, text=True).stdout.strip()
print(check())
if not check().endswith("True"):
    !uv pip install -q -p {PY} --reinstall torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
    print(check())
assert check().endswith("True"), "torch cannot see the GPU: is the accelerator set to GPU T4 x2?"

## 3. Ollama and NuExtract 8B Q8

Ollama gets the second GPU, the encoders the first, so the 6 GB swapping of the local setup never runs out of memory here.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import time, urllib.request
n_gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
LLM_GPU = "1" if n_gpus > 1 else "0"
ollama = subprocess.Popen(["ollama", "serve"], env={**os.environ, "CUDA_VISIBLE_DEVICES": LLM_GPU},
                          stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode())

In [ ]:
# Same Modelfile as locally (extractor/nuextract.Modelfile: ChatML template, temperature 0), pointed at the GGUF.
template = open(f"{EXT}/nuextract.Modelfile").read().split("\n", 1)[1]
repo, file = GGUF
path = subprocess.run([PY, "-c", f"from huggingface_hub import hf_hub_download as d; print(d({repo!r}, {file!r}))"],
                      capture_output=True, text=True, check=True).stdout.strip()
open(f"{WORK}/{MODEL}.Modelfile", "w").write(f"FROM {path}\n{template}")
subprocess.run(["ollama", "create", MODEL, "-f", f"{WORK}/{MODEL}.Modelfile"], check=True, capture_output=True)
!ollama list

## 4. Extractor server

The warm-up chapter also downloads GLiNER2 and BookNLP (~3 GB) and checks the whole pipeline before the tunnel opens.

In [ ]:
import json
server = subprocess.Popen([PY, "-m", "extractor", "serve", "--port", str(PORT)], cwd=EXT,
                          env={**os.environ, "ORION_LLM_MODEL": MODEL, "ORION_EXTRACTOR_TOKEN": TOKEN,
                               "CUDA_VISIBLE_DEVICES": "0"},
                          stdout=open(f"{WORK}/server.log", "w"), stderr=subprocess.STDOUT)
health = lambda base: urllib.request.urlopen(f"{base}/health", timeout=10).read().decode()
for _ in range(120):
    try:
        print(health(f"http://localhost:{PORT}")); break
    except Exception:
        assert server.poll() is None, open(f"{WORK}/server.log").read()[-3000:]
        time.sleep(2)

def extract(base, text):
    request = urllib.request.Request(f"{base}/extract", json.dumps({"story_id": "warmup", "text": text}).encode(),
                                     {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(request, timeout=900) as response:
        return json.load(response)

t = time.time()
doc = extract(f"http://localhost:{PORT}", open(f"{WORK}/orion/docs/e2e_story_gull_point.txt").read()[:1500])
print(f"warm-up {time.time() - t:.0f}s:", {k: len(v) for k, v in doc.items() if isinstance(v, list)})

## 5. ngrok tunnel

In [ ]:
!curl -fsSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz | tar xz -C /usr/local/bin
!ngrok config add-authtoken {NGROK_AUTHTOKEN} > /dev/null
tunnel = subprocess.Popen(["ngrok", "http", str(PORT), f"--url=https://{NGROK_DOMAIN}", "--log=stdout"],
                          stdout=open(f"{WORK}/ngrok.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        print("public:", health(f"https://{NGROK_DOMAIN}")); break
    except Exception:
        assert tunnel.poll() is None, open(f"{WORK}/ngrok.log").read()[-3000:]
        time.sleep(2)
print(f'backend/.env:\nEXTRACTOR_URL="https://{NGROK_DOMAIN}"\nEXTRACTOR_TOKEN="<ORION_EXTRACTOR_TOKEN>"')

## 6. Keep the session alive

A committed run ends after its last cell, so this one waits, printing the server's log every 10 minutes.

In [ ]:
end = time.time() + HOURS * 3600
while time.time() < end:
    for name, p in (("ollama", ollama), ("server", server), ("ngrok", tunnel)):
        assert p.poll() is None, f"{name} exited: see {WORK}/{name}.log"
    print(time.strftime("%H:%M"), "| server log:", *open(f"{WORK}/server.log").read().splitlines()[-3:], sep="\n  ",
          flush=True)
    time.sleep(600)